# Chess Tutor Web App – Codebase Walkthrough

## Overview

This notebook provides a detailed walkthrough of the project's architecture, file structure, and core logic. The system is designed with a strong separation of concerns, dividing responsibilities across frontend, orchestration backend, and compute backends.

The system is composed of five major components:

- `public/` → Frontend (UI and client-side interaction)
- `src/` → Authoritative backend (game state + orchestration)
- `engine_backend/` → Engine evaluation (Stockfish-based analysis)
- `playerbot_backend/` → Human-like bot move generation
- `tutor_backend/` → Move analysis and LLM-powered coaching feedback

The backend (`src/`) is the **single source of truth**, while the Python services act as **compute workers**.

---

## High-Level File Structure
```
project_root/
│
├── public/               # Frontend
│
├── src/                  # Backend (authoritative orchestrator)
│   ├── controllers/      # Request orchestration logic
│   ├── game/             # Game state + status
│   ├── routes/           # API endpoints
│   └── services/         # External service interfaces
│
├── engine_backend/       # Engine evaluation service (port 8000)
├── playerbot_backend/    # Human-like bot service (port 8001)
├── tutor_backend/        # Move coaching service (port 8002)
│
└── server.js             # Node.js frontend server (port 3000)
```

This structure ensures clean separation between:
- UI
- State management
- Request orchestration
- Heavy computation

---

## Frontend (`public/`)

The frontend is responsible for all user interaction and rendering.

### Responsibilities

- Render chess board and UI panels
- Capture user actions (moves, toggles, settings)
- Send API requests to backend
- Display:
  - Updated board state
  - Move history
  - Engine feedback
  - Tutor responses

### Key Principle

The frontend is **stateless with respect to game rules**. It does not validate moves or compute outcomes—it relies entirely on the backend.

### Toggle Persistence

The state of the Engine, Bot, Tutor, and Novice Mode toggles is saved to `localStorage` on every change and restored on page load. This means all active features survive a browser refresh without requiring the user to re-enable them manually.

### UI Controls

- **Engine toggle** – enables Stockfish analysis; shows the eval bar and best-move arrow on the board
- **Bot toggle** – starts the human-like playerbot as the opponent
- **ELO selector** – sets the target skill level for the bot (0–3000); persists across refreshes
- **Tutor toggle** – enables move-by-move coaching feedback after each player move
- **Novice Mode toggle** – modifies tutor output to use beginner-friendly language
- **Flip Board** – mirrors the board orientation

---

## Backend (`src/`) – Authoritative Orchestrator

The `src/` directory is the core of the system. It manages all state and coordinates every subsystem.

---

### Controllers (`controllers/`)

Controllers act as the **entry point for logic execution**. Each controller corresponds to a domain of responsibility:

- `board` → Board state queries and updates
- `move` → Move validation and execution
- `engine` → Engine evaluation requests
- `clock` → Time management and updates
- `stream` → Real-time updates (SSE)
- `tutor` → Enable/disable tutor and novice mode

### Responsibilities

- Receive parsed request data from routes
- Call into game state and services
- Aggregate results
- Return structured responses

### Example Flow

HTTP Request → Route → Controller → Game/Services → Response

Controllers do not store state—they orchestrate it.

---

### Game (`game/`)

The `game/` folder encapsulates all game-related state and logic.

#### Files

- `game_state` → Holds board state, FEN, move history
- `game_status` → Tracks game lifecycle (turn, checkmate, draw, etc.)

### Responsibilities

- Maintain authoritative game state
- Enforce turn order
- Detect game-ending conditions
- Provide clean interface for controllers

### Key Principle

This layer is **pure state + rules**, with no external dependencies.

---

### Routes (`routes/`)

Defines API endpoints exposed to the frontend.

#### Endpoints

- `/move` → Submit a move
- `/board` → Fetch board state
- `/engine` → Request evaluation
- `/clock` → Manage time
- `/stream` → Subscribe to updates (SSE)
- `/playerbot` → Start/stop the opponent bot
- `/tutor` → Enable/disable tutor; set novice mode

### Responsibilities

- Map HTTP endpoints to controllers
- Perform lightweight validation
- Keep routing logic separate from business logic

---

### Services (`services/`)

Services act as the **bridge between the backend and external compute systems**.

#### Components

- Engine service → communicates with `engine_backend`
- PlayerBot service → communicates with `playerbot_backend`
- Tutor service → communicates with `tutor_backend`; tracks `enabled` and `novice` state
- SSE service → streams updates to clients

### Responsibilities

- Handle outbound requests to Python services
- Normalize responses into backend-friendly format
- Manage communication protocols (HTTP / SSE)

### Key Insight

Services isolate external dependencies, allowing:
- Easier testing
- Cleaner controller logic
- Swappable backends

---

## Engine Backend (`engine_backend/`)

This service performs position evaluation using Stockfish.

### Responsibilities

- Accept FEN input
- Run engine analysis
- Return:
  - Best move
  - Top candidate lines
  - Evaluation scores

### Role in System

Provides **ground-truth evaluation** used by:
- The eval bar and best-move arrow in the UI
- The tutor system for move classification

---

## Player Bot Backend (`playerbot_backend/`)

This service generates human-like moves. Two bot modes are available, selected at game start.

---

### Human Bot (Default)

Instead of relying on a single engine, the system uses **three distinct engines**:

1. **Weak Maia** (ELO − 300)
   - Simulates lower-skill human play
   - Produces realistic mistakes and variability

2. **Same-strength Maia**
   - Matches target ELO distribution
   - Provides human-like decision-making

3. **ELO-limited Stockfish**
   - Provides tactical sharpness
   - Ensures strong moves appear in critical positions

---

### Policy-Based Engine Selection

A learned linear policy (`engine_policy_v1.npz`) determines **which engine to use per move**.

#### Inputs to Policy

- Position evaluation (centipawns)
- Position complexity (eval spread between top moves)
- Player ELO setting

#### Output

A softmax probability distribution over the three engines:

```
P(engine | position, elo)

Example:
  Weak Maia:    0.50
  Same Maia:    0.30
  Stockfish:    0.20
```

---

### Move Generation Process

1. Receive current FEN
2. Run quick Stockfish evaluation to get position features
3. Use policy to sample engine
4. Query selected engine for a move
5. Apply artificial think delay: `clamp(remaining_seconds * 0.03, 0.05s, 2.0s)`
6. Return chosen move

---

### Why This Works

- Weak Maia introduces realistic mistakes
- Strong engines handle tactical correctness
- Policy blending creates **human-like inconsistency**, critical for realistic training opponents

---

### Retrieval Bot

The second bot mode selects moves by searching a database of real human games rather than running a neural network or engine.

#### Data and Indexing

At startup, the retrieval model loads a CSV of real games and builds two indices:
- **Full FEN index** — maps exact board positions (including move counters) to game records
- **Partial key index** — maps piece placement + turn + castling + en passant (ignoring move counts) to game records

Each position is also encoded as a **785-dimensional vector**: 12 piece types × 64 squares (one-hot), active turn, and four castling rights bits. These vectors support cosine similarity search.

#### Retrieval Hierarchy

For a given position and target ELO, the bot searches in three levels and uses the first that returns results:

```
Level 1 – Exact FEN match       → ELO window ±100 (relaxes to ±200)
Level 2 – Partial key match     → ELO window ±100 (relaxes to ±200)
Level 3 – Cosine similarity     → Top 300 candidates, ELO window ±200
```

This prioritizes exact transpositions, falls back to structurally identical positions, and finally retrieves approximately similar positions.

#### Weighted Voting

Moves from all retrieved games are aggregated by weighted vote:

```
weight = w_elo * w_similarity * w_win * w_rated

w_elo        = exp(-|player_elo - target_elo| / 150)
w_similarity = cosine similarity (1.0 for levels 1 and 2)
w_win        = 1.15 if the player won that game, else 1.0
w_rated      = 1.05 if the game was rated, else 1.0
```

Only legal moves receive weight. Scores are normalized to a probability distribution; the top move is played and the top three moves with probabilities are returned to the frontend for display.

The same clock-proportional think delay used by the Human Bot is applied before the move is returned.

---

## Tutor Backend (`tutor_backend/`)

This service analyzes the move a player just made and returns a natural-language coaching explanation.

### Trigger

After each player move (not bot moves), the Node orchestrator calls `/tutor/analyze` with:
- `before_fen` – position before the move
- `after_fen` – position after the move
- `played_move` – the move in UCI notation
- `novice` – whether to use beginner-friendly language

### Analysis Pipeline

The `TutorAnalyzer` class runs the following steps before calling the LLM:

1. **Stockfish evaluation** (depth 12) on both positions
   - Centipawn scores before and after
   - Best move and principal variation (5 moves)
   - Delta from the mover's perspective → move classification

2. **Move classification**
   - Blunder: > 100 cp lost
   - Mistake: > 50 cp lost
   - Inaccuracy: > 20 cp lost
   - Fine: ≤ 20 cp lost

3. **Board facts** (pure python-chess, no extra engine calls)
   - Game phase (opening / middlegame / endgame)
   - Material balance for both sides
   - King locations and castling status

4. **Move facts** (pure python-chess)
   - Piece moved, from/to squares, capture (if any)
   - Whether the destination was already attacked by the opponent (walked into danger)
   - Which of the mover's pieces are left en prise (attacked and undefended) after the move
   - Whether the move gives check
   - Same breakdown for the engine's best move alternative

5. **Legal moves list** (SAN notation)
   - All legal moves available before the played move
   - Passed to the LLM to prevent it from suggesting illegal alternatives

### Grounding Philosophy

All tactical facts are computed directly from the position using python-chess and Stockfish. The LLM receives verified data and is instructed not to invent details beyond what is provided. This prevents hallucinated tactical explanations.

### Novice Mode

When `novice=True`, an additional instruction is appended to the user prompt asking the LLM to explain in beginner-friendly language and define any chess terms it uses. The system prompt remains unchanged.

### Response Delivery

The explanation is returned to the Node orchestrator, which broadcasts it to the frontend via SSE as a `tutorUpdate` event. The tutor panel displays it immediately.

---

### Follow-Up Question Feature

After `analyze()` completes, the full LLM message history — system prompt, grounding user message, and assistant response — is cached in `TutorAnalyzer._conv_messages`.

When the user submits a follow-up question via the frontend text field:

1. Frontend POSTs `{ question }` to `POST /api/tutor/followup`
2. Node.js controller forwards to `POST /tutor/followup` on the Python service
3. Python appends `{role: 'user', content: question}` to `_conv_messages` and calls the LLM
4. The response is appended to `_conv_messages` (enabling chained follow-ups)
5. Explanation is returned to the frontend and displayed in the tutor panel

The cache is replaced on every new `analyze()` call, so context resets automatically when the player makes a new move. The frontend hides the follow-up text field on every `setFen` event and shows it again after the next `tutorUpdate` arrives.

---

## End-to-End Request Flow

```
Frontend
  ↓
API Route (/move)
  ↓
Controller (move)
  ↓
Game State Update
  ↓
Services Layer
  ↓
┌──────────────┬──────────────┬────────────────┐
↓              ↓              ↓                ↓
Engine         PlayerBot      Tutor           SSE Stream
(Eval)         (Move)         (Explanation)   (Update)
└──────────────┴──────────────┴────────────────┘
  ↓
Controller aggregates response
  ↓
Frontend updates UI
```

---

## Real-Time Streaming (SSE)

The system uses Server-Sent Events (SSE) to push updates.

### Event Types

- `setFen` – new board position after a move; includes a `source` field (`'player'` or `'bot'`) so the frontend suppresses the tutor thinking indicator during bot moves
- `engineUpdate` – new Stockfish evaluation line
- `tutorUpdate` – coaching explanation from the tutor backend
- `flip` – board orientation change
- `select` – square selection highlight

### Benefit

- Lower latency than polling
- Simpler than WebSockets for this one-way use case
- Allows engine and tutor results to arrive independently as they complete

---

## Running the Application

The repository includes startup scripts that handle dependency installation, Stockfish resolution, and launching all four services automatically.

**Windows (PowerShell):**

```powershell
cd chess_tutor
.\start_all_services.ps1 -ApiKey "your_key_here"
```

**macOS / Linux (bash):**

```bash
cd chess_tutor
DUKE_API_KEY="your_key_here" bash run_system.sh
```

Both scripts create the Python virtual environment if missing, install all pip and npm dependencies, locate or install Stockfish, and launch each service. Then open `http://localhost:3000`.

To start services manually, run the following in four separate terminals with the Python venv activated:

```bash
# Terminal 1 – Frontend (port 3000)
cd chess_tutor
node server.js

# Terminal 2 – Engine backend (port 8000)
cd chess_tutor/engine_backend
uvicorn main:app --port 8000

# Terminal 3 – Playerbot backend (port 8001)
cd chess_tutor/playerbot_backend
uvicorn main:app --port 8001

# Terminal 4 – Tutor backend (port 8002)
cd chess_tutor/tutor_backend
uvicorn main:app --port 8002
```

---

## Key Takeaways

- `src/` is the authoritative backend and central coordinator
- Controllers orchestrate logic but do not store state
- Game layer owns all state and rules
- Services isolate external compute systems
- Engine backend provides optimal evaluation
- Player Bot backend supports two modes: a learned policy blending Maia and Stockfish for human-like play, and a retrieval model that votes across real human games
- Tutor backend grounds LLM explanations in verified engine and python-chess analysis
- Follow-up questions extend the same LLM conversation with full positional context
- All bots apply a clock-proportional think delay to simulate realistic pacing
- Toggle state persists across browser refreshes via `localStorage`
- The `source` field on `setFen` SSE events distinguishes player moves from bot moves, preventing the tutor panel from flashing during bot turns